# N11 · X-ray emission from the hot gas in halos

## Bremsstrahlung, the $\beta$-model, and the soft-X-ray–galaxy connection

**Module 3 — Multi-wavelength large-scale-structure cosmology (PhD onboarding).**
The X-ray emission of the hot gas in groups and clusters is the fourth observable of the thesis,
mapped by **eROSITA**. This notebook builds the emission physics and connects directly to the two
A&A 697 papers cited in the thesis proposal: the soft-X-ray–galaxy cross-correlation
(Comparat et al. 2025) and the simulation-based hot-CGM forward model (Shreeram et al. 2025).

**Key tools:** `astropy.constants`, `astropy.cosmology`, `numpy`, `scipy`.


### Learning objectives

1. Write the thermal bremsstrahlung (free–free) emissivity and recognise its exponential cut-off.
2. Build the isothermal $\beta$-model gas profile and project it to an X-ray surface brightness.
3. Compute the X-ray luminosity of a model halo from its emission measure.
4. Relate luminosity to mass via the self-similar and observed $L_X$–$M$ scaling.
5. Connect to the soft-X-ray stacking and projection effects studied in the thesis references.


## References

- Sarazin (1986), Rev. Mod. Phys. 58, 1 — X-ray emission of clusters
  [doi:10.1103/RevModPhys.58.1](https://doi.org/10.1103/RevModPhys.58.1).
- Cavaliere & Fusco-Femiano (1976), A&A 49, 137 — the $\beta$-model
  [ADS:1976A&A....49..137C](https://ui.adsabs.harvard.edu/abs/1976A%26A....49..137C).
- Smith et al. (2001), ApJ 556, L91 — APEC plasma emission code
  [arXiv:astro-ph/0106478](https://arxiv.org/abs/astro-ph/0106478).
- Pratt et al. (2009), A&A 498, 361 — REXCESS $L_X$–$M$ scaling
  [arXiv:0809.3784](https://arxiv.org/abs/0809.3784).
- **Comparat et al. (2025), A&A 697, A173** — soft-X-ray $\times$ galaxy cross-correlation
  [doi:10.1051/0004-6361/202554208](https://doi.org/10.1051/0004-6361/202554208).
- **Shreeram et al. (2025), A&A 697, A22** — simulation-based hot-CGM model, projection effects
  [arXiv:2409.10397](https://arxiv.org/abs/2409.10397).


## 1. Thermal bremsstrahlung

The dominant continuum process in the $10^7$–$10^8$ K intracluster/intragroup medium is thermal
bremsstrahlung. The frequency-dependent emissivity is (Sarazin 1986, eq. 5.4; Rybicki & Lightman 1979):

$$\epsilon_\nu^{\rm ff} = 6.8\times10^{-38}\,Z^2\,n_e\,n_i\,T^{-1/2}\,e^{-h\nu/k_BT}\,\bar g_{\rm ff}
\quad[\mathrm{erg\,s^{-1}\,cm^{-3}\,Hz^{-1}}],$$

with $T$ in K. Two features matter for X-ray surveys: the spectrum is roughly flat up to the
**exponential cut-off** at $h\nu\sim k_BT$ (so the cut-off energy is a thermometer), and integrating
over frequency gives the total free–free emissivity (Sarazin 1986, eq. 5.15):

$$\epsilon^{\rm ff} = 1.4\times10^{-27}\,T^{1/2}\,n_e\,n_i\,Z^2\,\bar g_B
\quad[\mathrm{erg\,s^{-1}\,cm^{-3}}].$$

In practice line emission is added with a plasma code such as APEC (Smith et al. 2001); here we use
the free–free continuum to keep the physics transparent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.constants import G, m_p, k_B
from astropy.cosmology import Planck18 as cosmo

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11, 'axes.grid': True,
                     'grid.alpha': 0.3})

# Bremsstrahlung spectral shape e^{-E/kT} (Gaunt factor ~ const for visualisation)
E = np.logspace(-1, 1.5, 400)            # photon energy [keV]
plt.figure()
for kT in [2.0, 5.0, 10.0]:              # keV
    eps_nu = np.exp(-E / kT)             # shape (flat * exp cut-off)
    plt.loglog(E, eps_nu, label=f'$k_BT$ = {kT:.0f} keV')
plt.axvspan(0.5, 2.0, color='gold', alpha=0.2, label='eROSITA soft band 0.5–2 keV')
plt.xlabel('photon energy [keV]'); plt.ylabel(r'$\epsilon_\nu/\epsilon_\nu(0)$ (shape)')
plt.title('Thermal bremsstrahlung continuum'); plt.legend()
plt.savefig('xray_bremsstrahlung_spectrum.png', dpi=130); plt.show()


## 2. The $\beta$-model and the X-ray surface brightness

A simple, widely used description of the gas density is the isothermal **$\beta$-model**
(Cavaliere & Fusco-Femiano 1976):

$$n_e(r) = n_0\left[1 + (r/r_c)^2\right]^{-3\beta/2}.$$

Because the X-ray surface brightness is the line-of-sight integral of the emissivity
$\propto n_e^2$, the $\beta$-model has a closed-form projection (Cavaliere & Fusco-Femiano 1976):

$$S_X(b) \propto \int n_e^2\,dl \;\propto\; \left[1 + (b/r_c)^2\right]^{-3\beta + 1/2}.$$

This analytic profile is what is fitted to observed cluster images and stacked galaxy fields.


In [ ]:
rc = 0.15          # core radius [Mpc]
beta = 0.65        # typical cluster value
b = np.logspace(-2, 0.5, 200)            # projected radius [Mpc]

SX = (1 + (b/rc)**2)**(-3*beta + 0.5)
plt.figure()
plt.loglog(b, SX / SX[0])
plt.xlabel('projected radius b [Mpc]'); plt.ylabel(r'$S_X(b)/S_X(0)$')
plt.title(rf'$\beta$-model surface brightness ($\beta$={beta}, $r_c$={rc} Mpc)')
plt.savefig('xray_beta_model_SX.png', dpi=130); plt.show()
print("outer slope d log SX / d log b ->", -6*beta + 1, "at large b")


## 3. X-ray luminosity from the emission measure

The (bremsstrahlung) luminosity is the volume integral of the emissivity,

$$L_X = \int \epsilon^{\rm ff}\,dV = 1.4\times10^{-27}\,T^{1/2}\,\bar g_B \int n_e\,n_i\,dV,$$

where $\int n_e n_i\,dV$ is essentially the **emission measure**. For our isothermal $\beta$-model we
integrate $n_e^2$ (taking $n_i\simeq n_e$) out to $R_{500}$.


In [ ]:
from scipy.integrate import quad

def R500_of_M(M500, z):
    rho_c = cosmo.critical_density(z)
    R3 = (3 * M500 / (4 * np.pi * 500 * rho_c)).to(u.Mpc**3)
    return (R3**(1/3)).to(u.Mpc)

M500 = 5e14 * u.Msun
z_cl = 0.1
R500 = R500_of_M(M500, z_cl)
kT_keV = 5.0
T_K = kT_keV * 1.160451e7                 # 1 keV = 1.16045e7 K
n0 = 1.0e-3                                # central electron density [cm^-3] (typical cluster)

# emission integral  4 pi int n_e^2 r^2 dr  out to R500, with r,R in cm
Mpc_cm = (1 * u.Mpc).to_value(u.cm)
R500_cm = R500.to_value(u.Mpc) * Mpc_cm
rc_cm = rc * Mpc_cm
def integrand(r_cm):
    ne = n0 * (1 + (r_cm/rc_cm)**2)**(-3*beta/2)
    return ne**2 * 4*np.pi * r_cm**2
EI, _ = quad(integrand, 0, R500_cm, limit=200)     # cm^-3
gB = 1.2                                            # frequency-averaged Gaunt factor (Sarazin 1986)
L_X = 1.4e-27 * np.sqrt(T_K) * gB * EI             # erg/s
print(f"R500 = {R500:.3f},  emission integral = {EI:.3e} cm^-3")
print(f"L_X (bremsstrahlung, <R500) = {L_X:.3e} erg/s")


## 4. The $L_X$–$M$ scaling relation

Self-similar collapse predicts $L_X \propto E(z)^{7/3} M^{4/3}$ for pure bremsstrahlung
(Sarazin 1986). Non-gravitational heating (AGN feedback, the very physics the thesis aims to
constrain) steepens the observed relation to $L_X\propto M^{\sim1.6}$ (Pratt et al. 2009). The slope
and scatter of $L_X$–$M$ are a direct window on feedback — and a nuisance to marginalise over in the
cosmological inference.


In [ ]:
M = np.logspace(13.5, 15.0, 50) * u.Msun
Ez = cosmo.efunc(z_cl)
# normalise both laws to our computed (M500, L_X) point for illustration
M0 = M500.to_value(u.Msun)
L_selfsim = L_X * (M.to_value(u.Msun)/M0)**(4/3)
L_observed = L_X * (M.to_value(u.Msun)/M0)**(1.6)

plt.figure()
plt.loglog(M.to_value(u.Msun), L_selfsim, label=r'self-similar $L\propto M^{4/3}$ (Sarazin 1986)')
plt.loglog(M.to_value(u.Msun), L_observed, '--', label=r'observed $L\propto M^{1.6}$ (Pratt+2009)')
plt.scatter([M0], [L_X], color='k', zorder=5, label='model cluster')
plt.xlabel(r'$M_{500}\;[M_\odot]$'); plt.ylabel(r'$L_X$ [erg/s]')
plt.title('X-ray luminosity–mass scaling'); plt.legend()
plt.savefig('xray_LX_M_scaling.png', dpi=130); plt.show()


## 5. The scaling slope as an autodiff derivative

A scaling-relation slope is a *logarithmic derivative*. Writing the self-similar luminosity as a
differentiable JAX function of mass, automatic differentiation returns the slope exactly. Self-similar
collapse gives $T\propto M^{2/3}E^{2/3}$, gas density $\propto E^2$, volume $\propto M E^{-2}$, and a
bremsstrahlung cooling function $\Lambda\propto T^{1/2}$, so $L_X\propto M^{4/3}E^{7/3}$
(Sarazin 1986). The autodiff slope should be exactly $4/3$ — and the gap to the *observed* slope
($\sim1.6$, Pratt et al. 2009) is the imprint of feedback the thesis sets out to measure.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

def logL_selfsim(lnM, z):
    M = jnp.exp(lnM)
    E = float(cosmo.efunc(z))          # background expansion at z (constant)
    T  = M**(2/3) * E**(2/3)           # self-similar temperature (arb. units)
    ne = E**2                          # gas density ~ Delta_c * rho_c(z)
    V  = M * E**(-2)                   # volume ~ M / (Delta_c rho_c)
    Lam = jnp.sqrt(T)                  # bremsstrahlung cooling function
    return jnp.log(ne**2 * Lam * V)

slope = float(jax.grad(logL_selfsim, argnums=0)(jnp.log(5e14), 0.1))
print(f"autodiff  d ln L_X / d ln M = {slope:.4f}   (self-similar 4/3 = {4/3:.4f})")
print(f"observed slope (Pratt+2009) ~ 1.6  ->  feedback excess = {1.6 - slope:+.2f}")


## 6. Connection to the thesis: stacking and projection effects

The thesis stacks the soft X-ray signal **around galaxies** to measure the mean hot-gas content as a
function of stellar mass (Comparat et al. 2025). Two complications, both central to the thesis
references, arise:

- **Projection.** The line-of-sight integral that gives $S_X(b)$ also picks up *correlated*
  large-scale structure and neighbouring halos, biasing the recovered profile. Shreeram et al. (2025)
  build a forward model from hydrodynamical simulations to quantify exactly this, and show the
  intrinsic profile slope $\beta$ correlates with stellar mass.
- **Mis-centring / satellites.** Mis-identified central galaxies dilute the stack, with contamination
  growing toward low stellar mass (Shreeram et al. 2025).

These are precisely the *astrophysical* parameters that must be inferred jointly with $\Omega_m$ and
$\sigma_8$ — the motivation for the multi-probe forward model of the thesis (see N12–N13).


## Exercises

1. **Cut-off thermometer.** From the bremsstrahlung spectrum, show that fitting the exponential
   cut-off energy recovers $k_BT$. Add Poisson noise to a mock spectrum and estimate the temperature.
2. **Emission-weighted vs spectroscopic T.** For a non-isothermal cluster $T(r)=T_0[1+(r/r_c)^2]^{-0.3}$,
   compute the emission-measure-weighted temperature and compare to $T_0$.
3. **Soft-band fraction.** Using the $e^{-E/kT}$ continuum, compute the fraction of photons emitted in
   the eROSITA 0.5–2 keV band for $k_BT = 1, 3, 8$ keV. Which halos are "soft"?
4. **Group regime.** Repeat the $L_X$ calculation for a galaxy *group* ($M_{500}=10^{13.5}M_\odot$,
   $k_BT\simeq1$ keV, $n_0$ lower). Comment on why groups are where feedback matters most — the regime
   of Comparat et al. (2025).


## Summary

- Hot-gas X-ray emission is bremsstrahlung (plus lines); the spectral cut-off measures $T$.
- The $\beta$-model gives an analytic gas profile and surface brightness $S_X\propto[1+(b/r_c)^2]^{-3\beta+1/2}$.
- $L_X$ follows from the emission measure; $L_X$–$M$ encodes feedback (self-similar $M^{4/3}$ → observed $M^{1.6}$).
- Autodiff of the self-similar model returns the $4/3$ slope exactly; the excess to the observed slope *is* the feedback signal.
- Stacking around galaxies (the thesis) is limited by projection and mis-centring — astrophysical
  parameters to infer jointly with cosmology.

**Next (N12):** combine galaxies, lensing and these gas probes into a multi-probe $N\times2$pt forecast.
